# 03 - Full Agent Workflow

This notebook demonstrates the complete Smart Home Agent using LangGraph.

## Learning Objectives

1. Understand LangGraph workflow structure
2. See how nodes process state step-by-step
3. Observe GraphRAG + LLM integration
4. Debug and trace agent reasoning

## Prerequisites

- Completed notebooks 01 and 02
- Neo4j running with seed data
- OpenAI API key configured in `.env`

## Setup

In [6]:
import sys
sys.path.insert(0, '..')

import os
import json
from dotenv import load_dotenv
load_dotenv('../.env')

# Verify API key is set
api_key = os.getenv("OPENAI_API_KEY")
if api_key:
    print(f"✓ OpenAI API Key: {api_key[:8]}...{api_key[-4:]}")
else:
    print("✗ OpenAI API Key not found!")
    print("  Set OPENAI_API_KEY in ../.env file")

✓ OpenAI API Key: sk-proj-...KpYA


In [7]:
# Import agent components
from src.agent import SmartHomeAgent, create_initial_state, state_summary
from src.agent.workflow import create_workflow, visualize_workflow
from src.graph.retriever import SmartHomeRetriever

print("✓ Imports successful!")

✓ Imports successful!


## 1. Understanding the Workflow

Let's visualize the agent's workflow structure.

In [8]:
# Print the workflow diagram
print(visualize_workflow())


```mermaid
graph TD
    START([Start]) --> parse[Parse Intent]
    parse --> retrieve[Retrieve Context]
    retrieve --> check{Check Sufficiency}

    check -->|Needs Clarification| ask[Ask Clarification]
    check -->|Sufficient| plan[Generate Plan]

    ask --> END1([End])

    plan --> counter[Increment Counter]
    counter --> validate{Validate Plan}

    validate -->|Valid| response[Generate Response]
    validate -->|Invalid & Retry| plan

    response --> END2([End])

    style START fill:#90EE90
    style END1 fill:#FFB6C1
    style END2 fill:#FFB6C1
    style check fill:#FFE4B5
    style validate fill:#FFE4B5
```



### Workflow Nodes Explained

| Node | Purpose | Input | Output |
|------|---------|-------|--------|
| `parse_intent` | Extract structured intent from text | user_input | parsed_intent |
| `retrieve_context` | Query Neo4j via GraphRAG | parsed_intent | retrieved_context |
| `check_sufficiency` | Decide if we have enough info | retrieved_context | needs_clarification |
| `generate_plan` | LLM creates action plan | intent + context | action_plan |
| `validate_plan` | Check plan validity | action_plan | validation_result |
| `generate_response` | Create user-friendly response | action_plan | final_response |

In [9]:
# Create the workflow and inspect it
workflow = create_workflow()
compiled = workflow.compile()

# Get graph info
graph = compiled.get_graph()
print("Workflow Nodes:")
for node in graph.nodes:
    print(f"  - {node}")

Workflow Nodes:
  - __start__
  - parse_intent
  - retrieve_context
  - check_sufficiency
  - ask_clarification
  - generate_plan
  - validate_plan
  - generate_response
  - increment_counter
  - __end__


## 2. Running the Agent (Basic)

Let's run the agent with a simple request.

In [ ]:
# Create the agent
agent = SmartHomeAgent(debug=False)

# Simple request
response = agent.run("Turn on the living room lights")

print("Request: Turn on the living room lights")
print("\nResponse:")
print(response)

## 3. Running with Trace

Let's see the reasoning steps.

In [ ]:
# Run with trace
response, trace = agent.run_with_trace("Make the living room cozy for movie night")

print("Request: Make the living room cozy for movie night")
print("\n📋 Reasoning Trace:")
print("-" * 50)
for i, step in enumerate(trace, 1):
    print(f"{i}. {step}")

print("\n💬 Response:")
print("-" * 50)
print(response)

## 4. Inspecting Full State

Let's look at the complete state after processing.

In [ ]:
# Get full state
final_state = agent.get_full_state("Set up the bedroom for sleep")

# Print summary
print(state_summary(final_state))

In [ ]:
# Inspect individual state components
print("\n🎯 Parsed Intent:")
print(json.dumps(final_state.get('parsed_intent', {}), indent=2))

In [ ]:
# Inspect action plan
print("\n📋 Action Plan:")
print(json.dumps(final_state.get('action_plan', {}), indent=2))

In [ ]:
# Inspect retrieved context
context = final_state.get('retrieved_context', '')
print("\n📚 Retrieved Context (first 1000 chars):")
print("-" * 50)
print(context[:1000])
if len(context) > 1000:
    print(f"\n... ({len(context) - 1000} more characters)")

## 5. Streaming Execution

Watch the agent process step by step.

In [ ]:
# Stream execution
print("Streaming: 'Dim the lights in the office'")
print("=" * 50)

for event in agent.run_streaming("Dim the lights in the office"):
    # Each event is a dict with node name as key
    for node_name, updates in event.items():
        print(f"\n📍 Node: {node_name}")
        
        # Show key updates
        if 'parsed_intent' in updates:
            goal = updates['parsed_intent'].get('goal', 'N/A')
            print(f"   → Parsed goal: {goal}")
        
        if 'retrieved_context' in updates:
            ctx_len = len(updates['retrieved_context'])
            print(f"   → Retrieved {ctx_len} chars of context")
        
        if 'action_plan' in updates:
            actions = updates['action_plan'].get('actions', [])
            print(f"   → Generated {len(actions)} actions")
        
        if 'final_response' in updates:
            print(f"   → Response: {updates['final_response'][:100]}...")

## 6. Testing Different Scenarios

Let's test various types of requests.

In [ ]:
# Test scenarios
test_requests = [
    "Turn on the kitchen light",
    "Set up for a party in the living room",
    "I want to focus in my office",
    "Good morning!",  # Scene-based
]

print("Testing Multiple Scenarios")
print("=" * 60)

for request in test_requests:
    print(f"\n📝 Request: \"{request}\"")
    print("-" * 40)
    
    try:
        response, trace = agent.run_with_trace(request)
        print(f"Steps: {len(trace)}")
        print(f"Response: {response[:150]}..." if len(response) > 150 else f"Response: {response}")
    except Exception as e:
        print(f"Error: {e}")

## 7. Understanding the Node Functions

Let's look at what each node does in detail.

In [ ]:
# Import individual nodes
from src.agent.nodes import (
    parse_intent,
    retrieve_context,
    check_sufficiency,
    generate_plan,
    validate_plan,
    generate_response,
)

# Create initial state
state = create_initial_state("Make the living room relaxing")

print("Initial State:")
print(f"  user_input: {state['user_input']}")

In [ ]:
# Step 1: Parse Intent
print("\n" + "=" * 50)
print("STEP 1: Parse Intent")
print("=" * 50)

result = parse_intent(state)
state.update(result)

print("\nExtracted Intent:")
print(json.dumps(state['parsed_intent'], indent=2))

In [ ]:
# Step 2: Retrieve Context
print("\n" + "=" * 50)
print("STEP 2: Retrieve Context (GraphRAG)")
print("=" * 50)

result = retrieve_context(state)
state.update(result)

print(f"\nStrategies used: {state['retrieval_metadata']['strategies_used']}")
print(f"Context length: {len(state['retrieved_context'])} characters")
print("\nContext preview:")
print(state['retrieved_context'][:500])

In [ ]:
# Step 3: Check Sufficiency
print("\n" + "=" * 50)
print("STEP 3: Check Sufficiency")
print("=" * 50)

result = check_sufficiency(state)
state.update(result)

print(f"\nNeeds clarification: {state.get('needs_clarification', False)}")
if state.get('clarification_question'):
    print(f"Question: {state['clarification_question']}")

In [ ]:
# Step 4: Generate Plan (if sufficient)
if not state.get('needs_clarification'):
    print("\n" + "=" * 50)
    print("STEP 4: Generate Plan")
    print("=" * 50)
    
    result = generate_plan(state)
    state.update(result)
    
    print("\nGenerated Action Plan:")
    print(json.dumps(state['action_plan'], indent=2))

In [ ]:
# Step 5: Validate Plan
if state.get('action_plan'):
    print("\n" + "=" * 50)
    print("STEP 5: Validate Plan")
    print("=" * 50)
    
    result = validate_plan(state)
    state.update(result)
    
    validation = state['validation_result']
    print(f"\nIs valid: {validation['is_valid']}")
    if validation['issues']:
        print(f"Issues: {validation['issues']}")
    if validation['suggestions']:
        print(f"Suggestions: {validation['suggestions']}")

In [ ]:
# Step 6: Generate Response
if state.get('action_plan'):
    print("\n" + "=" * 50)
    print("STEP 6: Generate Response")
    print("=" * 50)
    
    result = generate_response(state)
    state.update(result)
    
    print(f"\nFinal Response:")
    print(state['final_response'])

## 8. Examining the Prompts

Let's look at the prompts that guide the LLM.

In [ ]:
from src.agent.prompts import (
    INTENT_PARSER_PROMPT,
    ACTION_PLAN_PROMPT,
    RESPONSE_PROMPT,
)

print("Intent Parser System Prompt:")
print("=" * 50)
print(INTENT_PARSER_PROMPT.messages[0].prompt.template[:500])

In [ ]:
print("\nAction Plan System Prompt:")
print("=" * 50)
print(ACTION_PLAN_PROMPT.messages[0].prompt.template[:800])

## 9. Exercises

Try these exercises to deepen your understanding.

In [ ]:
# Exercise 1: Test a vague request
# Does the agent ask for clarification?

# Your code here:
# response = agent.run("Make it nice")


In [ ]:
# Exercise 2: Examine how a scene-based request is handled
# What retrieval strategies are used?

# Your code here:
# state = agent.get_full_state("Movie night mode")
# print(state['retrieval_metadata'])


In [ ]:
# Exercise 3: Compare the action plans for similar requests
# Are they consistent?

# Your code here:
# plan1 = agent.get_full_state("Dim the living room lights")['action_plan']
# plan2 = agent.get_full_state("Make the living room darker")['action_plan']


In [ ]:
# Exercise 4: Modify a prompt and see the difference
# (Advanced - edit src/agent/prompts.py)

# Ideas:
# - Make the response more formal/casual
# - Add energy-saving considerations
# - Include safety warnings


## 10. Summary

### What We Learned

1. **LangGraph Workflows**
   - Nodes are processing functions
   - State flows through the graph
   - Conditional edges enable branching

2. **Agent Architecture**
   - Parse → Retrieve → Plan → Validate → Respond
   - Each step is modular and testable
   - State accumulates context

3. **GraphRAG Integration**
   - Intent determines retrieval strategy
   - Context is structured for LLM reasoning
   - Multiple strategies can combine

4. **LLM Orchestration**
   - Prompts guide LLM behavior
   - Output parsing structures results
   - Validation ensures quality

### Key Takeaways

- **Separation of concerns**: Each node has one job
- **Explicit state**: Everything is inspectable
- **Flexible routing**: Conditional logic is clear
- **Testable**: Each component can be tested alone

## What's Next?

- Try the CLI: `python app.py`
- Add new scenes to the knowledge graph
- Customize prompts for different styles
- Add new retrieval strategies
- Implement real device control (simulation mode)